# Lesson 2.4: Real CSV aur ganda data

Ab tak data hum **khud code mein** likh rahe the, aur woh saaf tha. Real duniya mein data **CSV** mein aata hai (Kaggle, client ka Excel, database dump), aur usmein masle hote hain: missing values, duplicates, typo, ghalat values.

Ames wala sabaq:
```
Inspect → Understand → Investigate → Decide → Preprocess
```

Ye notebook do hisson mein hai:
- **Part A: Inspect.** Detective ban ke masle **dhoondna**, kuch theek nahi karna
- **Part B: Decide aur Fix.** Har masle ka **faisla**, phir code

`data/innings.csv` mein jaan boojh ke **6 qisam ke masle** chhupaye gaye hain.

In [1]:
import pandas as pd

---
# Part A: Inspect (detective kaam) 🔍

## A1: CSV padhna
```
notebook yahan hai:     cricket-ml/notebooks/07_pandas_cleaning.ipynb
..                  →   ek folder upar (cricket-ml/)
../data/innings.csv →   cricket-ml/data/innings.csv
```
`FileNotFoundError` aaye to 99% **path** ka masla hota hai.

CSV mein jahan do commas ke beech kuch nahi (`2,,23`), Pandas wahan **NaN** (missing) rakhta hai.

In [2]:
df = pd.read_csv("../data/innings.csv")
df

,match_date,player,opposition,venue,batting_position,balls,runs
0,2024-01-10,Babar,India,Karachi,3,30.0,45.0
1,2024-01-10,Rizwan,India,Karachi,1,50.0,67.0
2,2024-01-15,Fakhar,Australia,Lahore,2,60.0,89.0
3,2024-01-15,Babar,Australia,Lahore,3,15.0,12.0
4,2024-01-20,Rizwan,England,Dubai,1,28.0,34.0
5,2024-01-20,Fakhar,England,Dubai,2,NaN,23.0
6,2024-02-02,Babar,india,Karachi,3,3.0,0.0
7,2024-02-02,Rizwan,India,Karachi,1,40.0,51.0
8,2024-02-02,Rizwan,India,Karachi,1,40.0,51.0
9,2024-02-08,Fakhar,England,NaN,2,70.0,102.0


In [3]:
df.shape   # (15, 7)

(15, 7)

## A2: `info()` — types aur non-null ginti

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   match_date        15 non-null     str    
 1   player            15 non-null     str    
 2   opposition        15 non-null     str    
 3   venue             14 non-null     str    
 4   batting_position  15 non-null     int64  
 5   balls             14 non-null     float64
 6   runs              14 non-null     float64
dtypes: float64(2), int64(1), str(4)
memory usage: 972.0 bytes


### Natija
- `venue`, `balls`, `runs`: **14 non-null** (15 nahi) → har ek mein 1 value missing
- `balls` aur `runs` **float64** hain (`40.0`), jabke CSV mein `40` tha.
  **Wajah:** Pandas ka normal integer type **NaN store nahi kar sakta**, isliye jis column mein ek bhi NaN ho, woh float ban jata hai.
  > Bina wajah decimals dikhein → pehle NaN check karo.

## A3: Missing values

| Command | Kya batata hai |
|---|---|
| `df.isnull().sum()` | har column mein **kitni** values missing |
| `df[df.isnull().any(axis=1)]` | woh **poori rows** jinmein koi bhi value missing ho |

`any(axis=1)` = "har row mein, kya **koi bhi** column True hai?" (`axis=1` = har row ka jawab)

In [5]:
df.isnull().sum()

match_date          0
player              0
opposition          0
venue               1
batting_position    0
balls               1
runs                1
dtype: int64

In [6]:
df[df.isnull().any(axis=1)]

,match_date,player,opposition,venue,batting_position,balls,runs
5,2024-01-20,Fakhar,England,Dubai,2,NaN,23.0
9,2024-02-08,Fakhar,England,NaN,2,70.0,102.0
11,2024-02-15,Rizwan,Australia,Dubai,1,35.0,NaN


### Masle 1: Missing values (3)
| Row | Player | Kya missing |
|---|---|---|
| 5 | Fakhar, 2024-01-20 | `balls` |
| 9 | Fakhar, 2024-02-08 | `venue` |
| 11 | Rizwan, 2024-02-15 | `runs` |

## A4: Duplicates

`df.duplicated()` har row ke liye True/False: *"kya ye row upar kisi row ki **exact copy** hai?"*
- Default mein sirf **doosri copy** True
- `keep=False` → **dono copies** dikhao, taake compare kar sakein

In [7]:
df.duplicated().sum()

np.int64(1)

In [8]:
df[df.duplicated(keep=False)]

,match_date,player,opposition,venue,batting_position,balls,runs
7,2024-02-02,Rizwan,India,Karachi,1,40.0,51.0
8,2024-02-02,Rizwan,India,Karachi,1,40.0,51.0


### Masla 2: Duplicate row
Rizwan, 2024-02-02, India, 40 balls, 51 runs: **rows 7 aur 8 bilkul same**.

## A5: Na-mumkin values — `describe()` + cricket ki samajh

`min` aur `max` pe khaas nazar, aur poochein: **"kya ye cricket mein mumkin hai?"**

In [9]:
df.describe()

,batting_position,balls,runs
count,15.000000,14.000000,14.000000
mean,2.066667,33.285714,85.000000
std,0.883715,21.449148,153.724529
min,1.000000,-12.000000,0.000000
25%,1.000000,25.750000,25.000000
50%,2.000000,36.500000,48.000000
75%,3.000000,43.000000,64.000000
max,3.000000,70.000000,610.000000


### Masle 3 aur 4
- `runs` max = **610** → T20 mein na-mumkin (Babar, 2024-02-08)
- `balls` min = **−12** → negative balls na-mumkin (Babar, 2024-02-15)

Dekhein 610 ne kya kiya: runs ka **mean 85** aa gaya, jabke **median 48**. Ek ghalat value ne average bigaar diya.

Ye masle **kisi command ne "error" bol ke nahi** bataye. Inhein **domain knowledge** ne pakra. Isi liye cricket project chuna.

## A6: Categories saaf hain? — `value_counts()` aur `unique()`

In [10]:
df["opposition"].value_counts()

opposition
India        5
Australia    4
England      4
india        1
India        1
Name: count, dtype: int64

In [11]:
df["opposition"].unique()

<StringArray>
['India', 'Australia', 'England', 'india', 'India ']
Length: 5, dtype: str

### Masle 5 aur 6: Text ki ghaltiyan
`India` 5 dafa, `india` 1, `India ` 1. 7 kyun nahi? Kyunki **computer ke liye ye 3 alag strings** hain:
```
"India"    →  I-n-d-i-a
"india"    →  i-n-d-i-a      ← chhota i
"India "   →  I-n-d-i-a-␣    ← aakhir mein space
```
**ML ke liye khatarnak:** OneHotEncoder inhein **3 alag teams** samjhega, aur India ka data teen hisson mein bat jayega. Koi error nahi aata, model chupke se kharab hota hai.
> Encoding se pehle **har categorical column** pe `unique()` chalao.

---
## Part A ka khulasa: 6 masle

| # | Masla | Kahan | Kis command se |
|---|---|---|---|
| 1 | Missing values | balls, venue, runs | `isnull().sum()` |
| 2 | Duplicate row | Rizwan 2024-02-02 | `duplicated()` |
| 3 | 610 runs | Babar 2024-02-08 | `describe()` max + cricket samajh |
| 4 | −12 balls | Babar 2024-02-15 | `describe()` min + cricket samajh |
| 5 | `india` (chhota i) | opposition | `unique()` |
| 6 | `India ` (extra space) | opposition | `unique()` |

---
# Part B: Decide aur Fix

### Order kyun important hai?
```
1. Text saaf karo
2. Duplicates hatao
3. Na-mumkin values → NaN
4. Target (runs) missing → row hatao
5. Features ki missing values bharo
6. Final check + save
```
**Pehle text, phir duplicates:** agar ek copy mein `India` aur doosri mein `India ` hota, to `duplicated()` unhein **alag** samajhta. Saaf karne ke baad hi asal duplicates nazar aate hain.

## Fix 1: Text saaf karna
**Faisla:** `india`, `India`, `India ` teeno ek hi team hain.

| Hissa | Kaam |
|---|---|
| `.str` | Pandas mein text functions ka darwaza |
| `.strip()` | shuru aur aakhir ki spaces hatao |
| `.title()` | har lafz ka pehla letter capital: `india` → `India` |

In [12]:
df["opposition"] = df["opposition"].str.strip().str.title()
df["opposition"].unique()   # sirf 3 teams

<StringArray>
['India', 'Australia', 'England']
Length: 3, dtype: str

## Fix 2: Duplicates hatana
**Faisla:** T20 mein ek player **ek match mein ek hi dafa** batting karta hai. Same date + player + values = galti se do dafa entry. Hata do.

⚠️ Hamesha sochein: **"kya ye row do dafa ho sakti hai?"** Ball-by-ball data mein do bilkul ek jaisi rows (do dot balls) **asli** ho sakti hain.

In [13]:
print("Pehle:", df.shape)
df = df.drop_duplicates()
print("Baad: ", df.shape)   # 15 → 14

Pehle: (15, 7)
Baad:  (14, 7)


## Fix 3: Na-mumkin values → NaN
**Faisla:**
- −12 balls: shayad 12 tha? **Yaqeen nahi.**
- 610 runs: shayad 61 tha? Ye bhi **andaza**.

> **Professional rule: andaze se data mat badlo.** Client/source se confirm karo. Confirm na ho sake to value ko **NaN** bana do, phir missing value ki tarah handle karo.

`df.loc[condition, column] = value` = SQL ka `UPDATE ... WHERE ...`
(T20 mein koi batsman 200 tak nahi pohancha, isliye 200 mehfooz sanity limit hai.)

In [14]:
df.loc[df["balls"] < 0, "balls"] = None
df.loc[df["runs"] > 200, "runs"] = None

df.isnull().sum()   # missing: balls 2, runs 2, venue 1

match_date          0
player              0
opposition          0
venue               1
batting_position    0
balls               2
runs                2
dtype: int64

Missing values **3 se 5** ho gayin. Theek hai: ghalat values ke saath train karne se behtar hai **imaandari se maan lein** ke ye maloom nahi.

## Fix 4: Target missing → row hatao
**Faisla:** `runs` hamara **target (y)** hai. Jis row mein jawab hi nahi, us se model kya seekhega? Aur median se bharna = model ko **jhoota jawab** sikhana.

> **Rule: target ki missing values kabhi nahi bharte. Woh rows hata dete hain.**

`subset=["runs"]` = **sirf** woh rows hatao jinmein `runs` missing hai. Baaki NaN wali rows rehne do (unhein bharenge).

In [15]:
df = df.dropna(subset=["runs"])
print(df.shape)   # 14 → 12
df

(12, 7)


,match_date,player,opposition,venue,batting_position,balls,runs
0,2024-01-10,Babar,India,Karachi,3,30.0,45.0
1,2024-01-10,Rizwan,India,Karachi,1,50.0,67.0
2,2024-01-15,Fakhar,Australia,Lahore,2,60.0,89.0
3,2024-01-15,Babar,Australia,Lahore,3,15.0,12.0
4,2024-01-20,Rizwan,England,Dubai,1,28.0,34.0
5,2024-01-20,Fakhar,England,Dubai,2,NaN,23.0
6,2024-02-02,Babar,India,Karachi,3,3.0,0.0
7,2024-02-02,Rizwan,India,Karachi,1,40.0,51.0
9,2024-02-08,Fakhar,England,NaN,2,70.0,102.0
12,2024-02-15,Babar,Australia,Dubai,3,NaN,20.0


Do rows hatin:
- Rizwan 2024-02-15 (runs CSV mein **khali** tha)
- Babar 2024-02-08 (**610** wali row: Fix 3 ne NaN banaya, Fix 4 ne hata di)

Aur **−12 balls** wali row? **Nahi hati**, kyunki `balls` feature hai, target nahi. Use Fix 5 bharega.

## Fix 5a: Venue — domain knowledge se bharna
Fakhar (2024-02-08) ka venue missing hai. Usi match mein Babar bhi khela tha, lekin uski row (610 wali) Fix 4 mein hat chuki hai. Saboot ke liye **asal CSV** mein us match ko dekhte hain:

In [16]:
original = pd.read_csv("../data/innings.csv")
original[original["match_date"] == "2024-02-08"]

,match_date,player,opposition,venue,batting_position,balls,runs
9,2024-02-08,Fakhar,England,NaN,2,70.0,102.0
10,2024-02-08,Babar,England,Lahore,3,44.0,610.0


Babar ka venue **Lahore** hai. **Ek match ek hi ground pe hota hai**, to Fakhar ka venue bhi Lahore hai. Ye andaza nahi, **mantiq** hai.

(Sabaq: asal file kabhi overwrite na karne ka faida yahi hai, saboot hamesha wahan maujood rehta hai.)

In [17]:
df.loc[(df["match_date"] == "2024-02-08") & (df["venue"].isnull()), "venue"] = "Lahore"
df[df["match_date"] == "2024-02-08"]

,match_date,player,opposition,venue,batting_position,balls,runs
9,2024-02-08,Fakhar,England,Lahore,2,70.0,102.0


## Fix 5b: Balls — median se bharna
Balls ka koi saboot nahi, isliye **median** (outliers ka asar nahi hota, Ames jaisa).

In [18]:
median_balls = df["balls"].median()
print("Median balls:", median_balls)   # 34.0

df["balls"] = df["balls"].fillna(median_balls)

Median balls: 34.0


⚠️ **Professional note:** yahan median **poore data** se nikala. Real ML project mein ye **leakage** hai (test data ki values bhi median mein shamil). Sahi tareeqa:
```
Train/test split → median SIRF training data se → dono pe lagao
```
Scikit-learn mein ye `SimpleImputer` karta hai, aur **Pipeline** mein automatic sahi hota hai.

Aur yaad rahe: ye do bhari hui values (34, 34) baad mein scatter chart pe **ek jagah jama** nazar aayengi (Lesson 3).

## Fix 6: Final check aur save

In [19]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df["opposition"].unique())

match_date          0
player              0
opposition          0
venue               0
batting_position    0
balls               0
runs                0
dtype: int64
Duplicates: 0
<StringArray>
['India', 'Australia', 'England']
Length: 3, dtype: str


In [20]:
df.describe()   # balls: 3 se 70, runs: 0 se 102 → sab cricket mein mumkin

,batting_position,balls,runs
count,12.000000,12.000000,12.000000
mean,2.166667,35.583333,44.083333
std,0.834847,18.352401,30.651882
min,1.000000,3.000000,0.000000
25%,1.750000,27.250000,22.250000
50%,2.000000,34.000000,39.500000
75%,3.000000,42.500000,58.000000
max,3.000000,70.000000,102.000000


`index=False` → index (0, 1, 2...) ko extra column ke tor pe mat likho (warna agli dafa `Unnamed: 0` column aata hai).

**Asal file (`innings.csv`) kabhi overwrite nahi karte.** Saaf data hamesha **nayi file** mein.

In [21]:
df.to_csv("../data/innings_clean.csv", index=False)
df.shape   # (12, 7)

(12, 7)

---
## Khulasa: rows ka hisaab
```
15 rows se shuru
− 1  Fix 2: duplicate (Rizwan 2024-02-02 ki doosri copy)
− 1  Fix 4: runs CSV mein khali (Rizwan 2024-02-15)
− 1  Fix 4: 610 runs → NaN → hati (Babar 2024-02-08)
= 12 rows
```

## Target vs feature
| Missing kya hai? | Kya karte hain | Kyun |
|---|---|---|
| **Target** (`runs`) | row hatao | jawab hi nahi to model kya seekhega |
| **Feature** (`balls`, `venue`) | bharo (median, mode, domain knowledge) | baaki row ki information qeemti hai |

## Commands
| Kaam | Command |
|---|---|
| Padhna | `read_csv`, `info`, `describe` |
| Missing | `isnull().sum()`, `isnull().any(axis=1)`, `dropna(subset=)`, `fillna` |
| Duplicates | `duplicated(keep=False)`, `drop_duplicates` |
| Text | `.str.strip()`, `.str.title()`, `unique()`, `value_counts()` |
| Value badalna | `df.loc[condition, column] = value` |
| Save | `to_csv(index=False)` |